# 02 - Feature Selection (Final)

**Goal:** Select final model features using reproducible importance and filtering steps.

**Inputs:** Previous stage artifacts and configured data paths

**Outputs:** Versioned artifacts for the next stage


## Report-Ready Runbook
1. Update path/config variables
2. Run cells top-to-bottom
3. Capture final metrics/artifacts for README
4. Keep exploratory diagnostics in `notebooks/archive/` only


In [ ]:
## Imports
from pyspark import SparkContext
from pyspark.sql import SQLContext
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.feature import VectorAssembler
from sklearn.metrics import confusion_matrix
from sklearn.datasets import load_iris
import pandas as pd
from pyspark.sql.functions import col, month

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.ml.classification import RandomForestClassifier

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, VectorSlicer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline




In [ ]:
import pandas as pd
import numpy as np

feature_importances_magnitude = [vec for vec in feature_importances]
feature_df = pd.DataFrame({'Features': total_cols, 'Importance': feature_importances_magnitude})
feature_df = feature_df.sort_values(by='Importance', ascending=False)
feature_df

In [ ]:
from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

from pyspark.sql.functions import col, month

folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_flights_12m_feat_new.parquet"

df_flights_12m_feat = spark.read.parquet(file_path)

#both cat and numerical
df_flights_12m_feat = df_flights_12m_feat.repartition(1000)

train_12mo = df_flights_12m_feat.filter(month(col("FL_DATE")) < 9).cache() 
test_12mo = df_flights_12m_feat.filter(month(col("FL_DATE")) >= 9).cache() 

# Filter out invalid labels
train_12mo = train_12mo.filter(col("DEP_DELAY_GROUP") >= 0)
test_12mo = test_12mo.filter(col("DEP_DELAY_GROUP") >= 0)


In [ ]:

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel
from itertools import product

# Example inputs
label_col = "DEP_DELAY_GROUP"
time_col = "FL_DATE"
# categorical_cols = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HHMM','Time_of_Day','DIVERTED','DISTANCE_GROUP']

# numerical_cols = ['FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights']

# param_grid = {
#     'maxDepth': [5, 10],
#     'numTrees': [20, 50]
# }

param_grid = {
    'maxDepth': [5],
    'numTrees': [10]
}

# Load your preprocessed DataFrame `df` here

label_col = 'DEP_DELAY_GROUP'
# Compute class weights
#df_weighted, weights_dict = compute_class_weights(train_12mo, label_col='DEP_DELAY_GROUP')

# Run CV folds
folds = expanding_window_cv(train_12mo, time_col=time_col, n_splits=3)

for i, (train_df, val_df) in enumerate(folds):
    print(f"\n===== Fold {i+1} =====")
    # Reduce the number of partitions to optimize memory usage
    # train_df = train_df.repartition(128)
    # val_df = val_df.repartition(128)
    
    # # Cache the DataFrames to avoid recomputation
    # train_df.cache()
    # val_df.cache()
    
    #train_df, val_df, categorical_cols,numerical_cols, param_grid
    train_df_weighted, weights_dict = compute_class_weights(train_df, label_col='DEP_DELAY_GROUP')
    best_model, pipeline_model = grid_search(train_df_weighted, val_df, categorical_cols,numerical_cols, param_grid)
    params = best_model.extractParamMap()
    print(params)
    evaluate_model(best_model, pipeline_model, val_df)

In [ ]:
#test Random forest with class weights for each fold; additional enineered features; feature Importance scores with class weights and additional features

In [ ]:
categorical_columns = [col for col, dtype in df_flights_12m_feat.dtypes if dtype == "string"]
numerical_columns = [col for col, dtype in df_flights_12m_feat.dtypes if dtype in ("int", "double", "float","bigint")]

In [ ]:
#final list
#final list
categorical_columns = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HH','Time_of_Day','DIVERTED','DISTANCE_GROUP']

numerical_columns = ['FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights','avg_delay_by_airline'] 

total_columns = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HH','Time_of_Day','DIVERTED','DISTANCE_GROUP','FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights','avg_delay_by_airline']

In [ ]:
#4/8/25 random forest hyoer-parameter tuning
from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

from pyspark.sql.functions import col, month

#both cat and numerical
df_flights_12m_feat = df_flights_12m_feat.repartition(1000)

train_12mo = df_flights_12m_feat.filter(month(col("FL_DATE")) <= 6).cache() 
val_12mo = df_flights_12m_feat.filter(month(col("FL_DATE")).isin([7,8,9])).cache() 
test_12mo = df_flights_12m_feat.filter(month(col("FL_DATE")) > 9).cache() 

# Filter out invalid labels
train_12mo = train_12mo.filter(col("DEP_DELAY_GROUP") >= 0)
val_12mo = val_12mo.filter(col("DEP_DELAY_GROUP") >= 0)
test_12mo = test_12mo.filter(col("DEP_DELAY_GROUP") >= 0)

# Step 4: Encode categorical features
indexers = [StringIndexer(inputCol=col, outputCol=col+"_index", handleInvalid="skip") for col in categorical_columns if col != "DEP_DELAY_GROUP" or col!= 'DEP_TIME' or col != 'DEP_DELAY' or col != 'DEP_DELAY_NEW' or col!= 'dep_timestamp' or col != 'DEP_TIME_HHMM' or col != 'DEP_DEL15']

encoders = [OneHotEncoder(inputCol=col+"_index", outputCol=col+"_ohe") for col in categorical_columns if col != "DEP_DELAY_GROUP" or col!= 'DEP_TIME' or col != 'DEP_DELAY' or col != 'DEP_DELAY_NEW' or col!= 'dep_timestamp' or col != 'DEP_TIME_HHMM' or col != 'DEP_DEL15']

#  numerical features for scaling
num_assembler = VectorAssembler(inputCols=numerical_columns, outputCol="num_features", handleInvalid="skip")

# Standardize numerical features
# scaler = StandardScaler(inputCol="num_features", outputCol="num_features_scaled", withMean=True, withStd=True)


from pyspark.ml import Pipeline

# Assemble final feature vector (OHE categorical + scaled numerical)
# feature_cols = [col+"_ohe" for col in categorical_columns] + ["num_features_scaled"]

feature_cols = [col+"_index" for col in categorical_columns] + ["num_features"]
final_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

# Initialize Random Forest Model
# Create a Random Forest Classifier
rf = RandomForestClassifier(featuresCol="features", labelCol="DEP_DELAY_GROUP", numTrees=20, maxDepth=5, maxBins=2000)


# Step 9: Create pipeline
# pipeline = Pipeline(stages=indexers + [num_assembler, scaler, final_assembler, rf])
pipeline = Pipeline(stages=indexers + [num_assembler, final_assembler, rf])


#Random Grid Search
# --- Step 5: Define Random Grid ---
# paramGrid = ParamGridBuilder() \
#     .addGrid(rf.numTrees, [10, 20, 30]) \
#     .addGrid(rf.maxDepth, [2, 5, 10]) \
#     .build(numModels=3)  # choose N random combinations
#         # .addGrid(rf.maxBins, [64, 128, 256]) \

evaluator = MulticlassClassificationEvaluator(
    labelCol="DEP_DELAY_GROUP", predictionCol="prediction", metricName="f1")


# --- Step 5: Manual Tuning Loop ---
best_model = None
best_f1 = float('-inf')
best_params = None

# for param_map in paramGrid:
#     rf_tmp = rf.copy(param_map)
pipeline = Pipeline(stages=indexers + encoders + [num_assembler, final_assembler, rf])

model = pipeline.fit(train_12mo)
val_pipeline = Pipeline(stages=indexers + encoders + [num_assembler, final_assembler])
val_data = val_pipeline.fit(val_12mo)
predictions = model.transform(val_data)
f1 = evaluator.evaluate(predictions)

# print(f"Params: {param_map} => F1 = {f1:.4f}")

# if f1 > best_f1:
#     best_f1 = f1
#     best_model = model
#     # best_params = param_map

# print(f"\nBest Params: {best_params}")
# print(f"Best Val F1: {best_f1:.4f}")

# --- Step 6: Evaluate Best Model on Final Test Set ---
test_data = val_pipeline.fit(test_12mo)
test_predictions = model.transform(test_data)
test_f1 = evaluator.evaluate(test_predictions)
print(f"Final F1 on Oct–Dec Test Set: {test_f1:.4f}")

# --- Step 7: Top 10 Feature Importances ---
rf_model = model.stages[-1]
feature_importances = rf_model.featureImportances
print(feature_importances)

importances = rf_model.featureImportances.toArray()
final_features = model.stages[-2].getInputCols()

feature_scores = list(zip(total_cols, importances))
top_features = sorted(feature_scores, key=lambda x: x[1], reverse=True)[:10]

print("\nTop 10 Most Important Features:")
for feature, score in top_features:
    print(f"{feature}: {score:.4f}")

In [ ]:
import pandas as pd
import numpy as np
feature_importances = rf_model.featureImportances
feature_importances_magnitude = [vec for vec in feature_importances]
feature_df = pd.DataFrame({'Features': total_cols, 'Importance': feature_importances_magnitude})
feature_df = feature_df.sort_values(by='Importance', ascending=False)
feature_df

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline


In [ ]:
#final list
categorical_cols = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HH','Time_of_Day','DIVERTED','DISTANCE_GROUP']

numerical_cols = ['FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights','avg_delay_by_airline'] 

total_cols = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HH','Time_of_Day','DIVERTED','DISTANCE_GROUP','FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights','avg_delay_by_airline']


In [ ]:
#2 hr 4 min for first 2 folds
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel
from itertools import product
print("Start model training")
# Example inputs
label_col = "DEP_DELAY_GROUP"
time_col = "FL_DATE"
# categorical_cols = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HHMM','Time_of_Day','DIVERTED','DISTANCE_GROUP']

# numerical_cols = ['FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights']

param_grid = {
    'maxDepth': [5],
    'numTrees': [10]
}

# param_grid = {
#     'maxDepth': [5, 10],
#     'numTrees': [10, 20, 30, 50]
# }

# Load your preprocessed DataFrame `df` here

label_col = 'DEP_DELAY_GROUP'
# Compute class weights
#df_weighted, weights_dict = compute_class_weights(train_12mo, label_col='DEP_DELAY_GROUP')

# Run CV folds
folds = expanding_window_cv(train_12mo, time_col=time_col, n_splits=3)

scores = []
best_score = 0

for i, (train_df, val_df) in enumerate(folds):
    print(f"\n===== Fold {i+1} =====")
    # Reduce the number of partitions to optimize memory usage
    train_df = train_df.repartition(128)
    val_df = val_df.repartition(128)
    
    # Cache the DataFrames to avoid recomputation
    train_df.cache()
    val_df.cache()
    
    #train_df, val_df, categorical_cols,numerical_cols, param_grid 
    train_df_weighted, weights_dict = compute_class_weights(train_df, label_col='DEP_DELAY_GROUP')
    best_model, pipeline_model, metrics, f1 = grid_search(train_df_weighted, val_df, categorical_cols,numerical_cols, param_grid)

    #  # Evaluate on validation set
    # metrics = evaluate_model(best_model, pipeline_model, val_df)
    
    # You can use F1, accuracy, or another primary metric
    current_score = metrics['f1']  # replace 'f1' with the actual key your evaluate_model returns
    scores.append(current_score)
    print(f"Fold {i+1} F1 score: {current_score}")
    print("Params:", best_model.extractParamMap())
    print("Fold {i} Metrics: ", metrics)


# Take average of all scores
avg_score = np.average(scores)    
# Update best score and parameter set to reflect optimal dev performance
if avg_score > best_score:
    best_score = avg_score
    print(f'new best score of {best_score:.2f}')
else:
    print(f'Result was no better, score was {avg_score:.2f} with best F1 score {best_score:.2f}')

print("\n===== Best Model Across All Folds =====")
print("Best F1 score:", best_score)

In [ ]:
#3rd fold
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel
from itertools import product
print("Start model training")
# Example inputs
label_col = "DEP_DELAY_GROUP"
time_col = "FL_DATE"
# categorical_cols = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HHMM','Time_of_Day','DIVERTED','DISTANCE_GROUP']

# numerical_cols = ['FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights']

param_grid = {
    'maxDepth': [5],
    'numTrees': [10]
}

# param_grid = {
#     'maxDepth': [5, 10],
#     'numTrees': [10, 20, 30, 50]
# }

# Load your preprocessed DataFrame `df` here

label_col = 'DEP_DELAY_GROUP'
# Compute class weights
#df_weighted, weights_dict = compute_class_weights(train_12mo, label_col='DEP_DELAY_GROUP')

# Run CV folds
folds = expanding_window_cv(train_12mo, time_col=time_col, n_splits=3)

scores = []
best_score = 0

for i, (train_df, val_df) in enumerate(folds):
    if i == 2:
        print(f"\n===== Fold {i+1} =====")
        # Reduce the number of partitions to optimize memory usage
        # train_df = train_df.repartition(128)
        # val_df = val_df.repartition(128)
        
        # # Cache the DataFrames to avoid recomputation
        # train_df.cache()
        # val_df.cache()
        
        #train_df, val_df, categorical_cols,numerical_cols, param_grid
        train_df_weighted, weights_dict = compute_class_weights(train_df, label_col='DEP_DELAY_GROUP')
        best_model, pipeline_model, metrics, f1 = grid_search(train_df_weighted, val_df, categorical_cols,numerical_cols, param_grid)

        #  # Evaluate on validation set
        # metrics = evaluate_model(best_model, pipeline_model, val_df)
        
        # You can use F1, accuracy, or another primary metric
        current_score = metrics['f1']  # replace 'f1' with the actual key your evaluate_model returns
        scores.append(current_score)
        print(f"Fold {i+1} F1 score: {current_score}")
        print("Params:", best_model.extractParamMap())
        print("Fold {i} Metrics: ", metrics)


# Take average of all scores
avg_score = np.average(scores)    
# Update best score and parameter set to reflect optimal dev performance
if avg_score > best_score:
    best_score = avg_score
    print(f'new best score of {best_score:.2f}')
else:
    print(f'Result was no better, score was {avg_score:.2f} with best F1 score {best_score:.2f}')

print("\n===== Best Model Across All Folds =====")
print("Best F1 score:", best_score)

In [ ]:
#Summary: grid search hyper-parameter tuning with first fold of 3-fold cross validation; final F1-score: 0.7251 and for the rest of the 7 param combos after 5,10 the F1 score stabilized at 0.7245; best model: 5 maxdepth, 10 trees

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel
from itertools import product

# Example inputs
label_col = "DEP_DELAY_GROUP"
time_col = "FL_DATE"
# categorical_cols = ['OP_UNIQUE_CARRIER','Holiday_Encoded','ORIGIN_STATE_ABR','ORIGIN_AIRPORT_ID','DEST_STATE_ABR','DEST_AIRPORT_ID','QUARTER','DAY_OF_MONTH','DAY_OF_WEEK','MONTH','origin_type','origin_region','dest_type','dest_region','DEP_TIME_HHMM','Time_of_Day','DIVERTED','DISTANCE_GROUP']

# numerical_cols = ['FLIGHTS','DISTANCE','HourlyAltimeterSetting','HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature','HourlyWindDirection','HourlyWindSpeed','avg_delay_by_airport','avg_delay_byairport_airline','num_severe_delayed_flights','num_outgoing_flights']

# param_grid = {
#     'maxDepth': [5, 10],
#     'numTrees': [20, 50]
# }

param_grid = {
    'maxDepth': [5, 10],
    'numTrees': [10, 20, 30, 50]
}

# Load your preprocessed DataFrame `df` here

label_col = 'DEP_DELAY_GROUP'
# Compute class weights
#df_weighted, weights_dict = compute_class_weights(train_12mo, label_col='DEP_DELAY_GROUP')

# Run CV folds
folds = expanding_window_cv(train_12mo, time_col=time_col, n_splits=3)

scores = []
best_score = 0

for i, (train_df, val_df) in enumerate(folds):
    print(f"\n===== Fold {i+1} =====")
    # Reduce the number of partitions to optimize memory usage
    # train_df = train_df.repartition(128)
    # val_df = val_df.repartition(128)
    
    # # Cache the DataFrames to avoid recomputation
    # train_df.cache()
    # val_df.cache()
    
    #train_df, val_df, categorical_cols,numerical_cols, param_grid
    train_df_weighted, weights_dict = compute_class_weights(train_df, label_col='DEP_DELAY_GROUP')
    best_model, pipeline_model = grid_search(train_df_weighted, val_df, categorical_cols,numerical_cols, param_grid)

     # Evaluate on validation set
    metrics = evaluate_model(best_model, pipeline_model, val_df)
    
    # You can use F1, accuracy, or another primary metric
    current_score = metrics['f1']  # replace 'f1' with the actual key your evaluate_model returns
    scores.append(current_score)
    print(f"Fold {i+1} F1 score: {current_score}")
    print("Params:", best_model.extractParamMap())


   


    #  # Set best parameter set to current one for first fold
    #   if best_param_vals == None:
    #     best_param_vals = p



    # Take average of all scores
    avg_score = np.average(scores)    
    # Update best score and parameter set to reflect optimal dev performance
    if avg_score > best_score:
      best_score = avg_score
    #   best_parameters = param_print
    #   best_param_vals = p
      print(f'new best score of {best_score:.2f}')
    else:
      print(f'Result was no better, score was {avg_score:.2f} with best F1 score {best_score:.2f}')

print("\n===== Best Model Across All Folds =====")
print("Best F1 score:", best_score)
# print("Best Params:", best_params)

In [ ]:

feature_importances = best_model.featureImportances
feature_importances_magnitude = [vec for vec in feature_importances]
feature_df2 = pd.DataFrame({'Features': total_cols, 'Importance': feature_importances_magnitude})
feature_df2 = feature_df2.sort_values(by='Importance', ascending=False)
feature_df2

In [ ]:
#Optuna run #2: reduce maxBins 
import optuna
# import optuna_integration
#from optuna.integration.pyspark import SparkDistributedTrial
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pyspark.sql.functions as f 
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
#run optuna with spark
#from optuna.integration.spark import SparkTrialExecutor
from optuna import create_study

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import DoubleType

from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.storagelevel import StorageLevel
from itertools import product

## Conclusion
Document key outcomes from this stage (dataset shape, selected features, best params, or metrics) before committing.
